# NB15-T6 — Ensemble, verifiers, LM and the dev ladder (SPEC T6, CPU part)

Everything here reads the posterior caches: **no GPU**. Builds ladder rows 1–7 on Extra-dev with 90% paired
sentence-bootstrap intervals (decision 25), plus the secondary rows and diagnostics T6 asks for.

| Row | System |
| --- | --- |
| 1 | BASE, best checkpoint, greedy |
| 2 | Recipe v2, single best checkpoint (by T5 dev F1) |
| 3 | Recipe v2, 6-checkpoint ensemble (mean probability) |
| 4 | Row 3 + **primary verifier** (decision 29: S only, acoustic + position + error type; every D reverted) |
| 5 | Row 3 + n-gram LM rescoring (NB10 method, recall guard), no verifier |
| 6 | Stage-1 + primary verifier trained on stage-1 candidates (T3 caches) |
| 7 | Stage-1 greedy (B0) |

Secondary rows (reported, never chosen): the full-feature verifier on all candidates (NB15 design) and the
deletion-only rule. Verifier training is out-of-fold: fold A candidates from the run-B ensemble, fold B from
the run-A ensemble, plus the Iqra dev slice from the 6-checkpoint ensemble (SPEC T6). The **design is fixed**
(decision 29); only τ and the LM weight are tuned here, on Extra-dev. Never touches QuranMB.v2 or IqraEval_Test_GT.

**Run settings:** Accelerator **None** (CPU), Internet *On*.
Add data: `iqra-cache`, the NB11 output, the **NB14-T6 output** (this task's caches) and the **NB14 output**
(`post_stage1_*.pt`, for rows 6 and 7). Secrets: `HF_TOKEN`.

**After the run:** Save Version. Paste back the last cell (**PASTE BACK**).


In [ ]:
# ---- Shared install cell (identical in nb11-nb16) --------------------------------
# pip install git+... does not fetch submodules, and the official scorer is a pinned
# submodule, so clone recursively and install from the clone (SPEC decision 21).
import os, subprocess, sys
REPO_URL = "https://github.com/markeet04/Quran_phenome.git"
REPO_DIR = "/tmp/Quran_phenome"          # outside /kaggle/working: not saved as output
BRANCH = "main"

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")   # only if the repo is private
except Exception:
    pass
url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL

def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode:
        msg = r.stderr if not token else r.stderr.replace(token, "***")
        raise RuntimeError(f"git {args[0]} failed: {msg}")
    return r.stdout.strip()

if not os.path.isdir(REPO_DIR):
    _git("clone", "--recursive", "-q", "-b", BRANCH, url, REPO_DIR)
else:   # a re-run in the same session: move the existing clone to the latest commit
    _git("-C", REPO_DIR, "fetch", "-q", url, BRANCH)
    _git("-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD")
_git("-C", REPO_DIR, "submodule", "update", "--init", "--recursive", "-q")
REPO_COMMIT = _git("-C", REPO_DIR, "rev-parse", "HEAD")

# No datasets pin: Kaggle now runs Python 3.13, where datasets 2.21's dependencies break.
# Audio is decoded by iqra.data (soundfile on the raw bytes), so any datasets version works.
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR],
                   capture_output=True, text=True)
if r.returncode:
    raise RuntimeError("pip install failed:\n" + r.stdout[-1500:] + "\n" + r.stderr[-1500:])
sys.path.insert(0, f"{REPO_DIR}/src")
import warnings
warnings.filterwarnings("ignore")
import iqra.align
import datasets, numpy, pandas
print("python", sys.version.split()[0], "| datasets", datasets.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__)
print("repo commit:", REPO_COMMIT)
print("official scorer:", iqra.align.mdd_eval_dir())


In [ ]:
# ---- Environment, inputs -----------------------------------------------------------------------
import glob, json, time
import numpy as np
import pandas as pd
import torch, lightgbm
from datasets import load_dataset, load_from_disk
from iqra import data, ensemble as E, lm as LM, posteriors, scorer, verifier
from iqra.candidates import candidate_features, remove_insertions
from iqra.config import Config, find_cache_root, find_input
from iqra.vocab import Vocab

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
except Exception as e:
    raise RuntimeError("HF_TOKEN secret is required for the gated Iqra_Extra_IS26") from e
T_START = time.time()
cfg = Config()
cfg.paths.cache_root = find_cache_root()
SPLITS_DIR = find_input("splits.json", what="the NB11 output")
T6_DIR = find_input("nb14_t6_report.json", what="the NB14-T6 output (this task's caches)")
S1_DIR = find_input("post_stage1_extra_dev.pt", what="the NB14 output (stage-1 caches)")
OUT = "/kaggle/working/nb15_t6"
os.makedirs(OUT, exist_ok=True)
vocab = Vocab.load(cfg.paths.vocab)
TOK2ID, ID2TOK, BLANK = vocab.tok2id, vocab.id2tok, vocab.blank_id
splits = data.load_splits(f"{SPLITS_DIR}/splits.json")
truth_dev = pd.read_csv(f"{SPLITS_DIR}/truth_extra_dev.csv", keep_default_na=False)
CLUSTERS = dict(zip(truth_dev["ID"], truth_dev["Reference_phn"]))          # decision 25
t6_report = json.load(open(f"{T6_DIR}/nb14_t6_report.json"))
report = {"repo_commit": REPO_COMMIT, "lightgbm": lightgbm.__version__, "nb14_t6_commit": t6_report["repo_commit"]}
print("caches:", T6_DIR, "| stage-1:", S1_DIR, "| lightgbm", lightgbm.__version__)


In [ ]:
# ---- Load caches; labels; ensembles -------------------------------------------------------------
caches, metas, member_info = {}, [], {}
for path in sorted(glob.glob(f"{T6_DIR}/post_*.pt")):
    meta, clips = posteriors.read_cache(path)
    caches.setdefault(meta["checkpoint"], {})[meta["split"]] = clips
    metas.append(meta)
    member_info[meta["checkpoint"]] = {"run": meta["run"], "step": int(meta["step"]), "t5_f1": float(meta["t5_f1"])}
posteriors.check_compatible(metas)
assert metas[0]["vocab_hash"] == vocab.hash
by_run = {r: sorted([n for n, i in member_info.items() if i["run"] == r], key=lambda n: -member_info[n]["t5_f1"])
          for r in E.EXPECTED_RUNS}
assert len(by_run["A"]) == 3 and len(by_run["B"]) == 3 and len(by_run["BASE"]) == 1, by_run
for r, names in by_run.items():                     # out-of-fold: each run cached only on its other fold
    for n in names:
        folds = {s for s in caches[n] if s.startswith("fold_")}
        assert folds == ({E.VERIFIER_FOLD[r]} if r in E.VERIFIER_FOLD else set()), (n, folds)
SIX = by_run["A"] + by_run["B"]
print("members:", {n: member_info[n] for n in SIX + by_run["BASE"]})

extra = load_dataset(cfg.datasets.extra, split="train", token=HF_TOKEN)
ex_ids = [str(x) for x in list(extra["id"])]
assert ex_ids == splits["extra"]["ids_by_row"]
ex_ref = dict(zip(ex_ids, [str(x) for x in list(extra["phoneme_ref"])]))
ex_mis = dict(zip(ex_ids, [str(x) for x in list(extra["phoneme_mis"])]))
train_arrow = load_from_disk(cfg.paths.train_arrow)
dev_phon = list(train_arrow["dev"]["phoneme_ref"])
dev_ref = {i: str(dev_phon[r]) for i, r in zip(splits["iqra_dev_slice"]["ids"], splits["iqra_dev_slice"]["rows"])}

def expected_said(split, cid):
    if split == "iqra_dev_slice":                  # correct speech: every flag is a false alarm
        return dev_ref[cid].split(), dev_ref[cid].split()
    return ex_ref[cid].split(), ex_mis[cid].split()

def ens(names, split):
    sub = {n: caches[n][split] for n in names}
    bad = E.frames_agree(sub, names)
    assert not bad, (split, bad[:3])
    return E.ensemble_split(sub, names, ID2TOK, BLANK)

t0 = time.time()
ENS = {"extra_dev": ens(SIX, "extra_dev"), "iqra_dev_slice": ens(SIX, "iqra_dev_slice"),
       "fold_b": ens(by_run["A"], "fold_b"), "fold_a": ens(by_run["B"], "fold_a")}   # out-of-fold
ENS_A3, ENS_B3 = ens(by_run["A"], "extra_dev"), ens(by_run["B"], "extra_dev")
S1 = {s: posteriors.read_cache(f"{S1_DIR}/post_stage1_{s}.pt")[1]
      for s in ("extra_dev", "fold_a", "fold_b", "iqra_dev_slice")}
print({s: len(v) for s, v in ENS.items()}, f"| ensembles built in {(time.time() - t0) / 60:.1f} min")


In [ ]:
# ---- Scoring helpers; rows 1, 2, 3, 7 and the T5 check (ensemble vs BASE) -----------------------
LADDER, COUNTS = {}, {}

def pred_from_tokens(tok_by_clip):
    return pd.DataFrame({"ID": list(tok_by_clip), "Prediction": [" ".join(t) for t in tok_by_clip.values()]})

def score(key, label, tok_by_clip, **extra_info):
    pred = pred_from_tokens(tok_by_clip)
    assert set(pred["ID"]) == set(truth_dev["ID"])
    pred.to_csv(f"{OUT}/pred_{key}_extra_dev.csv", index=False)
    res = scorer.official_score(truth_dev, pred, keep_dir=f"{OUT}/score_{key}")
    counts = scorer.per_clip_counts(truth_dev, pred)
    assert int(counts["TR"].sum()) == res["TR"] and int(counts["FR"].sum()) == res["FR"]
    ci = scorer.bootstrap_f1_interval(counts, CLUSTERS)
    COUNTS[key] = counts
    LADDER[key] = {"system": label, "precision": res["precision"], "recall": res["recall"], "f1": res["f1"],
                   "ci90": [round(ci["lo"], 4), round(ci["hi"], 4)], "TR": res["TR"], "FR": res["FR"],
                   "FA": res["FA"], **extra_info}
    print(f"{key:28s} F1 {res['f1']:.4f} [{ci['lo']:.4f}, {ci['hi']:.4f}]  P {res['precision']:.4f}  "
          f"R {res['recall']:.4f}  {label}")
    return res

def gain(new, base):
    g = scorer.paired_bootstrap_gain(COUNTS[base], COUNTS[new], CLUSTERS)
    return {"gain": round(g["gain"], 4), "ci90": [round(g["lo"], 4), round(g["hi"], 4)],
            "excludes_zero": g["excludes_zero"], "n_sentences": g["n_clusters"]}

tokens = lambda entries: {c: e["tokens"] for c, e in entries.items()}
best_single = max(SIX, key=lambda n: member_info[n]["t5_f1"])
score("row1_base", f"BASE {by_run['BASE'][0]}, greedy", tokens(caches[by_run["BASE"][0]]["extra_dev"]))
score("row2_best_single", f"recipe v2, best single checkpoint {best_single}", tokens(caches[best_single]["extra_dev"]))
score("row3_ensemble6", "recipe v2, 6-checkpoint ensemble", tokens(ENS["extra_dev"]))
score("row7_stage1", "stage-1 checkpoint-5000, greedy (B0)", tokens(S1["extra_dev"]))
score("info_A_best", f"run A alone, best checkpoint {by_run['A'][0]}", tokens(caches[by_run["A"][0]]["extra_dev"]))
score("info_B_best", f"run B alone, best checkpoint {by_run['B'][0]}", tokens(caches[by_run["B"][0]]["extra_dev"]))
score("info_A_ensemble3", "run A alone, 3-checkpoint ensemble", tokens(ENS_A3))
score("info_B_ensemble3", "run B alone, 3-checkpoint ensemble", tokens(ENS_B3))
GAINS = {"row3_vs_row1 (T5 done-when: ensemble beats BASE)": gain("row3_ensemble6", "row1_base"),
         "row3_vs_row2 (ensembling)": gain("row3_ensemble6", "row2_best_single")}
for k, v in GAINS.items():
    print(k, v)

# Decision 22 check on the ensemble: every insertion kept vs every insertion removed (report only)
exp_dev = dict(zip(truth_dev["ID"], truth_dev["Reference_phn"].str.split()))
noins = {c: remove_insertions(exp_dev[c], e["tokens"]) for c, e in ENS["extra_dev"].items()}
score("check_ensemble_no_insertions", "row 3 with every insertion removed (decision 22 check)", noins)
ins = gain("check_ensemble_no_insertions", "row3_ensemble6")
print("insertions removed vs kept:", ins, ">>> REPORT: removing insertions scored higher" if ins["gain"] > 0 else "")
report["insertion_check"] = ins


In [ ]:
# ---- Candidate tables (ensemble decodes; out-of-fold) ---------------------------------------------
def table(split, entries, tag):
    rows = []
    for cid, entry in entries.items():
        exp, said = expected_said(split, cid)
        rows += candidate_features(cid, exp, entry, TOK2ID, ID2TOK, BLANK, said=said)
    df = pd.DataFrame(rows)
    df["split"] = split
    df["sentence"] = [" ".join(expected_said(split, c)[0]) for c in df["clip"]]      # CV group
    df.to_parquet(f"{OUT}/candidates_{tag}_{split}.parquet", index=False)
    return df

t0 = time.time()
TAB = {s: table(s, ENS[s], "ens") for s in ("fold_a", "fold_b", "iqra_dev_slice", "extra_dev")}
assert set(TAB["fold_a"]["clip"]) <= set(splits["extra"]["fold_a"]) and set(TAB["fold_b"]["clip"]) <= set(splits["extra"]["fold_b"])
summary = {s: {"candidates": len(d), "real": int(d["label"].sum()), "S": int((d["op"] == "S").sum()),
               "D": int((d["op"] == "D").sum()), "real_S": int(d[d["op"] == "S"]["label"].sum()),
               "real_D": int(d[d["op"] == "D"]["label"].sum())} for s, d in TAB.items()}
for s, v in summary.items():
    print(f"{s:15s} {v}")
print(f"built in {(time.time() - t0) / 60:.1f} min")
report["candidates_ensemble"] = summary
train_df = pd.concat([TAB["fold_a"], TAB["fold_b"], TAB["iqra_dev_slice"]], ignore_index=True)
dev_df = TAB["extra_dev"]
assert not set(train_df["clip"]) & set(dev_df["clip"]), "Extra-dev leaked into verifier training"
HEARD = tokens(ENS["extra_dev"])


In [ ]:
# ---- Row 4: primary verifier (decision 29), tau on Extra-dev ---------------------------------------
def fit_primary(train, dev, heard, tag):
    booster, enc, cv = verifier.train_primary(train, cfg.verifier)
    s_rows = train[train["op"].isin(verifier.PRIMARY_OPS)].reset_index(drop=True)
    y = s_rows["label"].astype(int).to_numpy()
    is_extra = s_rows["split"].isin(["fold_a", "fold_b"]).to_numpy()
    cv_info = {"auc_pooled": round(cv["auc"], 4), "auc_extra_rows": round(verifier.auc(y[is_extra], cv["oof"][is_extra]), 4),
               "p_at_r90_extra_rows": round(verifier.precision_at_recall(y[is_extra], cv["oof"][is_extra]), 4),
               "n_rounds": int(np.median(cv["best_iterations"])), "train_S_rows": len(s_rows), "train_S_real": int(y.sum())}
    booster.save_model(f"{OUT}/verifier_primary_{tag}.txt")
    probs = verifier.predict_primary(booster, enc, dev)
    sweep = verifier.sweep_tau(dev, probs, truth_dev, heard, list(cfg.verifier.tau_grid), revert_deletions=True)
    sweep.to_csv(f"{OUT}/tau_sweep_primary_{tag}.csv", index=False)
    tau = verifier.robust_tau(sweep, cfg.verifier.robust_tau_band)
    new = verifier.apply(dev, probs, exp_dev, heard, tau, revert_deletions=True)
    # the fast counts must equal the official scorer at the chosen tau
    exact = float(sweep.loc[sweep["tau"] == tau, "f1"].iloc[0])
    return booster, enc, probs, sweep, tau, new, cv_info, exact

booster4, enc4, probs4, sweep4, tau4, new4, cv4, exact4 = fit_primary(train_df, dev_df, HEARD, "ensemble")
print("CV (S rows, grouped by sentence):", cv4)
print(sweep4[["tau", "precision", "recall", "f1"]].round(4).to_string(index=False))
r4 = score("row4_ens_primary_verifier", f"row 3 + primary verifier (tau {tau4})", new4, tau=tau4, cv=cv4)
assert round(exact4, 4) == r4["f1"], (exact4, r4["f1"])
GAINS["row4_vs_row3 (T6 done-when: >= 0.01, CI excludes 0)"] = gain("row4_ens_primary_verifier", "row3_ensemble6")
print(GAINS["row4_vs_row3 (T6 done-when: >= 0.01, CI excludes 0)"])

kept4 = dev_df.assign(prob=probs4, kept=(dev_df["op"] == "S") & (probs4 >= tau4))
by_type = []
for t, g in kept4.groupby("err_type"):
    k = g[g["kept"]]
    by_type.append({"err_type": t, "candidates": len(g), "real": int(g["label"].sum()),
                    "precision_before": round(float(g["label"].mean()), 3), "kept": len(k),
                    "precision_after": round(float(k["label"].mean()), 3) if len(k) else None,
                    "real_kept": int(k["label"].sum())})
by_type = pd.DataFrame(by_type).sort_values("candidates", ascending=False)
print(by_type.to_string(index=False))
report["row4_per_error_type"] = by_type.to_dict(orient="records")
report["clip_final_deletions"] = {"extra_dev_ensemble": E.clip_final_deletions(dev_df),
                                  "verifier_training_ensemble": E.clip_final_deletions(train_df)}
print("clip-final deletions:", report["clip_final_deletions"])


In [ ]:
# ---- Secondary rows: full-feature verifier on all candidates (NB15 design); deletion-only rule --------
cv_full = verifier.cross_validate(train_df, list(train_df["sentence"]), cfg.verifier)
n_full = int(np.median(cv_full["best_iterations"]))
b_full, enc_full = verifier.train(train_df, n_full, cfg.verifier)
probs_full = verifier.predict(b_full, enc_full, dev_df)
sweep_full = verifier.sweep_tau(dev_df, probs_full, truth_dev, HEARD, list(cfg.verifier.tau_grid))
sweep_full.to_csv(f"{OUT}/tau_sweep_full.csv", index=False)
tau_full = verifier.robust_tau(sweep_full, cfg.verifier.robust_tau_band)
score("secondary_full_verifier", f"row 3 + full-feature verifier, all candidates (tau {tau_full})",
      verifier.apply(dev_df, probs_full, exp_dev, HEARD, tau_full), tau=tau_full,
      cv={"auc_pooled": round(cv_full["auc"], 4), "n_rounds": n_full})
ones = np.where(dev_df["op"] == "S", 1.0, np.nan)          # keep every S, revert every D
score("secondary_deletion_only", "row 3, every deletion reverted, every substitution kept",
      verifier.apply(dev_df, ones, exp_dev, HEARD, 0.5, revert_deletions=True))
GAINS["secondary_full_vs_row3"] = gain("secondary_full_verifier", "row3_ensemble6")
GAINS["secondary_deletion_only_vs_row3"] = gain("secondary_deletion_only", "row3_ensemble6")
GAINS["row4_vs_deletion_only (what the S verifier adds)"] = gain("row4_ens_primary_verifier", "secondary_deletion_only")
for k in list(GAINS)[-3:]:
    print(k, GAINS[k])


In [ ]:
# ---- Row 5: n-gram LM rescoring of the plain ensemble decode (NB10 method), alpha on Extra-dev -------
dev_canon = set(truth_dev["Reference_phn"])
lines = [str(x) for x in list(train_arrow["train"]["phoneme_ref"])]
lines += [ex_ref[c] for f in ("fold_a", "fold_b") for c in splits["extra"][f]]
corpus, n_removed = LM.build_corpus(lines, exclude=dev_canon, vocab_symbols=vocab.phonemes())
t0 = time.time()
lm = LM.NgramLM(order=5, k=0.1).train(corpus)
print(f"LM corpus {len(corpus)} lines ({n_removed} removed: equal to an Extra-dev sentence) | "
      f"vocab {lm.V} | trained in {time.time() - t0:.0f} s")
cands = {c: LM.nbest(e["logits"].numpy(), ID2TOK, BLANK) for c, e in ENS["extra_dev"].items()}
assert all(cands[c][0][1] == HEARD[c] for c in cands), "N-best list must start with the greedy decode"
rows5 = []
for alpha in (0.0, 0.5, 1.0, 2.0, 4.0, 7.0, 12.0):
    pred = pred_from_tokens(LM.rescore(cands, lm, alpha))
    m = scorer.summed_metrics(scorer.per_clip_counts(truth_dev, pred))
    rows5.append({"alpha": alpha, **{k: round(v, 4) for k, v in m.items()}})
sweep5 = pd.DataFrame(rows5)
sweep5.to_csv(f"{OUT}/lm_alpha_sweep.csv", index=False)
print(sweep5.to_string(index=False))
choice = LM.choose_alpha(rows5, base_recall=LADDER["row3_ensemble6"]["recall"])
alpha5 = choice["alpha"]
score("row5_ens_lm", f"row 3 + n-gram LM (alpha {alpha5}, recall guard)", LM.rescore(cands, lm, alpha5),
      alpha=alpha5, corpus_lines=len(corpus), removed_dev_lines=n_removed)
GAINS["row5_vs_row3"] = gain("row5_ens_lm", "row3_ensemble6")
print("row5 vs row3:", GAINS["row5_vs_row3"], "| alpha 0 means no recall-safe setting beat the plain decode")


In [ ]:
# ---- Row 6: stage-1 + primary verifier trained on stage-1 candidates (T3 caches) ---------------------
TAB1 = {s: table(s, S1[s], "stage1") for s in ("fold_a", "fold_b", "iqra_dev_slice", "extra_dev")}
train1 = pd.concat([TAB1["fold_a"], TAB1["fold_b"], TAB1["iqra_dev_slice"]], ignore_index=True)
heard1 = tokens(S1["extra_dev"])
_, _, _, sweep6, tau6, new6, cv6, exact6 = fit_primary(train1, TAB1["extra_dev"], heard1, "stage1")
r6 = score("row6_stage1_primary_verifier", f"stage-1 + primary verifier (tau {tau6})", new6, tau=tau6, cv=cv6)
assert round(exact6, 4) == r6["f1"], (exact6, r6["f1"])
GAINS["row6_vs_row7"] = gain("row6_stage1_primary_verifier", "row7_stage1")
print("CV:", cv6, "| row6 vs row7:", GAINS["row6_vs_row7"])


In [ ]:
# ---- The dev ladder ----------------------------------------------------------------------------------
order = ["row1_base", "row2_best_single", "row3_ensemble6", "row4_ens_primary_verifier", "row5_ens_lm",
         "row6_stage1_primary_verifier", "row7_stage1", "secondary_full_verifier", "secondary_deletion_only",
         "info_A_best", "info_B_best", "info_A_ensemble3", "info_B_ensemble3", "check_ensemble_no_insertions"]
ladder = pd.DataFrame([{"key": k, **{c: LADDER[k][c] for c in ("system", "precision", "recall", "f1", "ci90")}}
                       for k in order])
ladder.to_csv(f"{OUT}/dev_ladder.csv", index=False)
with pd.option_context("display.width", 200, "display.max_colwidth", 70):
    print(ladder.round(4).to_string(index=False))
g43 = GAINS["row4_vs_row3 (T6 done-when: >= 0.01, CI excludes 0)"]
t6_done = g43["gain"] >= 0.01 and g43["ci90"][0] > 0
t5_done = GAINS["row3_vs_row1 (T5 done-when: ensemble beats BASE)"]["ci90"][0] > 0
final = "row4" if LADDER["row4_ens_primary_verifier"]["f1"] >= LADDER["row5_ens_lm"]["f1"] else "row5"
print(f"T5 done-when (ensemble beats BASE, CI excludes 0): {'PASS' if t5_done else 'FAIL'}")
print(f"T6 done-when (verifier adds >= 0.01 over ensemble, CI excludes 0): {'PASS' if t6_done else 'FAIL'}")
print(f"decision 16, provisional (T7 freezes it): final system = {final} by dev F1")
report.update({"ladder": LADDER, "gains": GAINS, "t5_done_when": bool(t5_done), "t6_done_when": bool(t6_done),
               "final_system_provisional": final, "members": member_info, "best_single": best_single,
               "minutes": round((time.time() - T_START) / 60, 1)})
with open(f"{OUT}/nb15_t6_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=1, default=str)


In [ ]:
# ---- PASTE BACK ----------------------------------------------------------------------------------------
print(json.dumps(report, ensure_ascii=False, indent=1, default=str))
print("files:", sorted(os.listdir(OUT)))
